In [ ]:
import pandas as pd
import numpy as np

## 2. Xây dựng mô hình Decision Tree Regressor từ đầu

Các thành phần chính:

- **Node**: lưu thông tin của từng nút trong cây.
- **DecisionTreeRegressor**: quản lý toàn bộ quá trình xây dựng và dự đoán bằng cây quyết định.
- Sử dụng phương pháp đệ quy để chia dữ liệu thành các nút con.
- Điều kiện dừng gồm:
  - Đạt độ sâu tối đa (`max_depth`).
  - Số lượng mẫu nhỏ hơn `min_samples_split`.
  - Phương sai của biến mục tiêu bằng 0.
- Giá trị dự đoán tại nút lá được tính bằng trung bình của các mẫu trong nút đó.

In [2]:
# CÀI ĐẶT CÂY QUYẾT ĐỊNH 

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       
        self.threshold = threshold   
        self.left = left             
        self.right = right           
        self.value = value           

    def is_leaf_node(self):
        return self.value is not None

class DecisionTreeRegressor:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        # 1. HÀM FIT: Nhận dữ liệu và bắt đầu mọc cây
        X_array = X.values if isinstance(X, pd.DataFrame) else X
        y_array = y.values if isinstance(y, pd.Series) else y
        self.root = self._grow_tree(X_array, y_array)

    def _grow_tree(self, X, y, depth=0):
            n_samples, n_feats = X.shape

            # Điều kiện biên để dừng phân tách cây
            if (depth >= self.max_depth or 
                n_samples < self.min_samples_split or 
                np.var(y) == 0.0):
                leaf_value = np.mean(y) # Giá trị lá là trung bình của y
                return Node(value=leaf_value)

           # 
            feat_idxs = np.arange(n_feats) 

            # Tìm kiếm điểm chia tốt nhất (Best Split) trên toàn bộ không gian đặc trưng
            best_feat, best_thresh = self._best_split(X, y, feat_idxs)

            # Tạo nhánh con đệ quy
            left_idxs = np.argwhere(X[:, best_feat] <= best_thresh).flatten()
            right_idxs = np.argwhere(X[:, best_feat] > best_thresh).flatten()
            
            if len(left_idxs) == 0 or len(right_idxs) == 0:
                return Node(value=np.mean(y))

            left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
            right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
            return Node(feature=best_feat, threshold=best_thresh, left=left, right=right)

    def _best_split(self, X, y, feat_idxs):
        # 3. HÀM BEST_SPLIT: Tìm điểm cắt giảm phương sai nhiều nhất
        best_gain = -1
        split_idx, split_thresh = 0, 0
        current_variance = np.var(y) * len(y) # Tổng phương sai của node hiện tại

        for feat_idx in feat_idxs: # Duyệt qua từng đặc trưng
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            
            # Lấy toàn bộ giá trị unique
            thresholds = np.unique(X_column)

            for thresh in thresholds: # duyệt qua từng điểm chia 
                left_idxs = np.argwhere(X_column <= thresh).flatten() # Tìm chỉ số của các mẫu thuộc nhánh trái
                right_idxs = np.argwhere(X_column > thresh).flatten()

                # bỏ qua split lỗi nếu một trong hai nhánh không có mẫu nào
                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue

                # Tính phương sai của nhánh trái và phải
                var_l = np.var(y[left_idxs])
                var_r = np.var(y[right_idxs]) # Tính phương sai của nhánh trái và phải
                n_l, n_r = len(left_idxs), len(right_idxs) # Số lượng mẫu trong nhánh trái và phải

                variance_reduction = current_variance - ((n_l * var_l) + (n_r * var_r)) # công thức giảm phương sai

                # Cập nhật split tốt nhất nếu tìm thấy
                if variance_reduction > best_gain:
                    best_gain = variance_reduction
                    split_idx = feat_idx
                    split_thresh = thresh

        return split_idx, split_thresh

    def predict(self, X):
        # 4. HÀM PREDICT: Dự đoán hàng loạt
        X_array = X.values if isinstance(X, pd.DataFrame) else X
        return np.array([self._traverse_tree(x, self.root) for x in X_array])

    def _traverse_tree(self, x, node):
        # 5. HÀM TRAVERSE_TREE: Duyệt từng dòng dữ liệu từ gốc đến lá
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

## 3. Huấn luyện và đánh giá mô hình cơ bản

Dữ liệu được chia thành:

- **Tập huấn luyện (80%)**: dùng để xây dựng mô hình.
- **Tập kiểm tra (20%)**: dùng để đánh giá khả năng dự đoán.

Sau khi huấn luyện, mô hình được đánh giá bằng các chỉ số:

- **MAE (Mean Absolute Error)**: sai số tuyệt đối trung bình.
- **MSE (Mean Squared Error)**: sai số bình phương trung bình.
- **RMSE (Root Mean Squared Error)**: căn bậc hai của MSE.
- **R^2 (R-squared)**: hệ số xác định.


In [ ]:
df_train = pd.read_csv('retail_train_80.csv')
target_col = 'sales_amount_log'

X_train = df_train.drop(columns=[target_col])
y_train = df_train[target_col]

df_test = pd.read_csv('retail_test_20.csv')
X_test = df_test.drop(columns=[target_col])
y_test = df_test[target_col]

print(f"Kích thước Train: {X_train.shape} | Kích thước Test: {X_test.shape}")
# CELL 4: HUẤN LUYỆN & TÍNH TOÁN ĐÁNH GIÁ

print("Đang huấn luyện mô hình ")
model = DecisionTreeRegressor(max_depth=10, min_samples_split=2)
model.fit(X_train, y_train)
print("Huấn luyện thành công!")

# dự đoán trên tập train
y_train_pred = model.predict(X_train)
# Dự đoán test
y_pred = model.predict(X_test)

# Đánh giá bằng công thức toán học (Numpy)
mae = np.mean(np.abs(y_test - y_pred))
mse = np.mean((y_test - y_pred) ** 2)
rmse = np.sqrt(mse)

# R² TRAIN

ss_res_train = np.sum((y_train - y_train_pred) ** 2)
ss_tot_train = np.sum((y_train - np.mean(y_train)) ** 2)

r2_train = 1 - (ss_res_train / ss_tot_train)

# R² Score
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)

r2 = 1 - (ss_res / ss_tot)

print("\n--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---")
print(f"1. MAE (Mean Absolute Error)     : {mae:.4f}")
print(f"2. MSE (Mean Squared Error)      : {mse:.4f}")
print(f"3. RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"4. R² Train                      : {r2_train:.4f}")
print(f"5. R² Test                       : {r2:.4f}")

Kích thước Train: (96000, 79) | Kích thước Test: (24000, 79)
Đang huấn luyện mô hình 
Huấn luyện thành công!

--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---
1. MAE (Mean Absolute Error)     : 0.4317
2. MSE (Mean Squared Error)      : 0.2434
3. RMSE (Root Mean Squared Error): 0.4933
4. R² Train                      : 0.7832
5. R² Test                       : 0.7564


## 5. Thử nghiệm điều chỉnh siêu tham số (Hyperparameter Tuning)

Để tìm cấu hình phù hợp cho mô hình Decision Tree Regressor, nhóm tiến hành thay đổi giá trị `max_depth`.

Tham số `max_depth` quyết định độ sâu tối đa của cây quyết định:

- Giá trị nhỏ giúp giảm hiện tượng overfitting nhưng có thể làm mô hình học chưa đủ.
- Giá trị lớn giúp mô hình học nhiều chi tiết hơn nhưng dễ dẫn đến overfitting.

Nhóm thực hiện thử nghiệm với các giá trị:

- max_depth = 10
- max_depth = 20
- max_depth = 30

Sau đó so sánh kết quả dựa trên các chỉ số MAE, MSE, RMSE và R² Score.

In [7]:

model_depth20 = DecisionTreeRegressor(
    max_depth=20,
    min_samples_split=2
)

# Train model
model_depth20.fit(X_train, y_train)

# Predict
y_train_pred_20 = model_depth20.predict(X_train)
y_pred_20 = model_depth20.predict(X_test)
# Evaluation
ss_res_train_20 = np.sum((y_train - y_train_pred_20) ** 2)
ss_tot_train_20 = np.sum((y_train - np.mean(y_train)) ** 2)

r2_train_20 = 1 - (ss_res_train_20 / ss_tot_train_20)


mae_20 = np.mean(np.abs(y_test - y_pred_20))
mse_20 = np.mean((y_test - y_pred_20) ** 2)
rmse_20 = np.sqrt(mse_20)

ss_res_20 = np.sum((y_test - y_pred_20) ** 2)
ss_tot_20 = np.sum((y_test - np.mean(y_test)) ** 2)

r2_20 = 1 - (ss_res_20 / ss_tot_20)

print("\n--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---")
print(f"1. MAE (Mean Absolute Error)     : {mae_20:.4f}")
print(f"2. MSE (Mean Squared Error)      : {mse_20:.4f}")
print(f"3. RMSE (Root Mean Squared Error): {rmse_20:.4f}")
print(f"4. R² Train                      : {r2_train_20:.4f}")
print(f"5. R² Test                       : {r2_20:.4f}")





--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---
1. MAE (Mean Absolute Error)     : 0.4939
2. MSE (Mean Squared Error)      : 0.4445
3. RMSE (Root Mean Squared Error): 0.6667
4. R² Train                      : 0.9450
5. R² Test                       : 0.5552


In [8]:
model_depth30 = DecisionTreeRegressor(
    max_depth=30,
    min_samples_split=2
)

# Train model
model_depth30.fit(X_train, y_train)

# Predict
y_pred_30 = model_depth30.predict(X_test)
y_train_pred_30 = model_depth30.predict(X_train)

# Evaluation
mae_30 = np.mean(np.abs(y_test - y_pred_30))
mse_30 = np.mean((y_test - y_pred_30) ** 2)
rmse_30 = np.sqrt(mse_30)

ss_res_train_30 = np.sum((y_train - y_train_pred_30) ** 2)
ss_tot_train_30 = np.sum((y_train - np.mean(y_train)) ** 2)

r2_train_30 = 1 - (ss_res_train_30 / ss_tot_train_30)

ss_res_30 = np.sum((y_test - y_pred_30) ** 2)
ss_tot_30 = np.sum((y_test - np.mean(y_test)) ** 2)

r2_30 = 1 - (ss_res_30 / ss_tot_30)

print("\n--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---")
print(f"1. MAE (Mean Absolute Error)     : {mae_30:.4f}")
print(f"2. MSE (Mean Squared Error)      : {mse_30:.4f}")
print(f"3. RMSE (Root Mean Squared Error): {rmse_30:.4f}")
print(f"4. R² Train                      : {r2_train_30:.4f}")
print(f"5. R² Test                       : {r2_30:.4f}")



--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---
1. MAE (Mean Absolute Error)     : 0.5088
2. MSE (Mean Squared Error)      : 0.4953
3. RMSE (Root Mean Squared Error): 0.7038
4. R² Train                      : 0.9996
5. R² Test                       : 0.5043


In [10]:
# Lưu kết quả test để nhóm đánh giá
result = pd.DataFrame({
    'y_test': y_test.values,
    'y_pred': y_pred
})

result.to_csv('prediction_result.csv', index=False)

print("Đã lưu file prediction_result.csv")

Đã lưu file prediction_result.csv


In [11]:
# Thêm cột dự đoán vào dataframe test gốc
df_test['y_pred'] = y_pred

# Lưu ra file CSV mới
df_test.to_csv('retail_test_with_prediction.csv', index=False)

print("Đã lưu file retail_test_with_prediction.csv")

Đã lưu file retail_test_with_prediction.csv
